# 196. Agent Context Engineering：怎样压缩上下文并按需恢复证据？

> **面试问题：如何在固定窗口下保留不可违反的约束、任务状态与可回查证据，并避免摘要版本漂移或关键事实丢失？**

## 先给结论

不要把 Agent 面试题答成框架 API：先定义状态、动作、权限、预算、版本和可判定的终态，再讨论 prompt、模型和并发扩展。下面用受控内存数据手写最小协议；小规模断言只证明实现合同，不代表线上模型效果、权限体系或安全等级。

## 一手资料

- [MemGPT](https://arxiv.org/abs/2310.08560)
- [Lost in the Middle](https://arxiv.org/abs/2307.03172)
- [RAG](https://arxiv.org/abs/2005.11401)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "assertions", "production": "isolation-and-audit"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "audit" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：压缩不等于删除历史

上下文窗口有限时，正确目标不是把文本尽量变短，而是为当前任务保留系统约束、未完成计划、关键事实及其来源。摘要必须有覆盖范围和版本，且关键事实应允许回查原始证据。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class Message:  # 执行本行的状态、计算或校验逻辑。
    message_id: str  # 执行本行的状态、计算或校验逻辑。
    role: str  # 执行本行的状态、计算或校验逻辑。
    text: str  # 执行本行的状态、计算或校验逻辑。
    pinned: bool = False  # 执行本行的状态、计算或校验逻辑。
history = [Message("m1", "system", "不得执行付款", True), Message("m2", "user", "查询订单 o-1"), Message("m3", "tool", "订单状态 paid")]  # 执行本行的状态、计算或校验逻辑。
assert history[0].pinned is True  # 执行本行的状态、计算或校验逻辑。
assert history[1].message_id == "m2"  # 执行本行的状态、计算或校验逻辑。
assert len(history) == 3  # 执行本行的状态、计算或校验逻辑。


## 2. token 预算：先明确上限与每段成本

生产中应使用实际 tokenizer 计数；教学代码用空格数近似，只说明预算算法接口。预算必须预留给 system prompt、工具结果、当前用户输入和模型输出，否则压缩后仍可能在调用时超窗。


In [ ]:
def estimate_tokens(text):  # 执行本行的状态、计算或校验逻辑。
    return len(text.split())  # 执行本行的状态、计算或校验逻辑。
def context_cost(messages):  # 执行本行的状态、计算或校验逻辑。
    return sum(estimate_tokens(message.text) for message in messages)  # 执行本行的状态、计算或校验逻辑。
assert estimate_tokens("a b c") == 3  # 执行本行的状态、计算或校验逻辑。
assert context_cost(history) == 5  # 执行本行的状态、计算或校验逻辑。
assert context_cost(history) <= 12  # 执行本行的状态、计算或校验逻辑。


## 3. 保留策略：不可丢弃的约束优先于最近闲聊

不要简单使用最近 N 条消息。这里先保留 pinned 消息和最新用户请求，再在剩余预算内挑选最近证据；生产系统还会引入任务状态、实体覆盖、会话阶段和策略风险。


In [ ]:
def select_context(messages, budget):  # 执行本行的状态、计算或校验逻辑。
    pinned = [message for message in messages if message.pinned]  # 执行本行的状态、计算或校验逻辑。
    selected = list(pinned)  # 执行本行的状态、计算或校验逻辑。
    for message in reversed(messages):  # 执行本行的状态、计算或校验逻辑。
        if message not in selected and context_cost(selected + [message]) <= budget:  # 执行本行的状态、计算或校验逻辑。
            selected.append(message)  # 执行本行的状态、计算或校验逻辑。
    return sorted(selected, key=lambda message: messages.index(message))  # 执行本行的状态、计算或校验逻辑。
selected = select_context(history, 8)  # 执行本行的状态、计算或校验逻辑。
assert history[0] in selected  # 执行本行的状态、计算或校验逻辑。
assert history[-1] in selected  # 执行本行的状态、计算或校验逻辑。
assert context_cost(selected) <= 8  # 执行本行的状态、计算或校验逻辑。


## 4. 摘要：输出事实、来源和覆盖范围，而不是神秘文本

真正的语义摘要可以由模型生成，但外部合同应保持确定：它覆盖哪些 message id、提炼哪些事实、使用哪个 prompt/model 版本。这里用规则抽取模拟 summary，避免把教学重点藏进一次 API 调用。


In [ ]:
def summarize(messages, summary_version):  # 执行本行的状态、计算或校验逻辑。
    facts = [message.text for message in messages if message.role == "tool"]  # 执行本行的状态、计算或校验逻辑。
    return {"covers": [message.message_id for message in messages], "facts": facts, "summary_version": summary_version}  # 执行本行的状态、计算或校验逻辑。
summary = summarize(history[1:], "summary-v1")  # 执行本行的状态、计算或校验逻辑。
assert summary["covers"] == ["m2", "m3"]  # 执行本行的状态、计算或校验逻辑。
assert summary["facts"] == ["订单状态 paid"]  # 执行本行的状态、计算或校验逻辑。
assert summary["summary_version"] == "summary-v1"  # 执行本行的状态、计算或校验逻辑。


## 5. 重建：摘要不能覆盖原始证据的身份

恢复上下文时应把 pinned 约束、最新请求、摘要和可按需检索的原文分开拼接。工具证据的 message id 仍要保留，防止摘要把过期事实伪装成当前事实。


In [ ]:
def rebuild_prompt(pinned, latest, summary):  # 执行本行的状态、计算或校验逻辑。
    return {"pinned": [item.text for item in pinned], "latest": latest.text, "summary": summary, "evidence_ids": summary["covers"]}  # 执行本行的状态、计算或校验逻辑。
prompt = rebuild_prompt([history[0]], history[1], summary)  # 执行本行的状态、计算或校验逻辑。
assert prompt["pinned"] == ["不得执行付款"]  # 执行本行的状态、计算或校验逻辑。
assert prompt["latest"] == "查询订单 o-1"  # 执行本行的状态、计算或校验逻辑。
assert prompt["evidence_ids"] == ["m2", "m3"]  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：版本或覆盖范围不匹配时重新压缩

摘要 prompt、模型或事实抽取规则变化后，旧摘要不应和新版本静默混用。生产系统也应在摘要引用的原文被删除、权限改变或实体冲突时使其失效。


In [ ]:
def validate_summary(summary, expected_version, allowed_ids):  # 执行本行的状态、计算或校验逻辑。
    return summary["summary_version"] == expected_version and set(summary["covers"]).issubset(set(allowed_ids))  # 执行本行的状态、计算或校验逻辑。
assert validate_summary(summary, "summary-v1", ["m2", "m3"])  # 执行本行的状态、计算或校验逻辑。
assert not validate_summary(summary, "summary-v2", ["m2", "m3"])  # 执行本行的状态、计算或校验逻辑。
assert not validate_summary(summary, "summary-v1", ["m2"])  # 执行本行的状态、计算或校验逻辑。


## 7. 按需检索：压缩后仍要能找回关键原文

摘要降低窗口成本，却可能遗漏细节。因此需要把原文放在可检索存储中，并通过 message id/权限过滤取回。这里用词重叠作极简检索；线上应替换为版本化索引和 ACL 过滤。


In [ ]:
def retrieve(query, messages):  # 执行本行的状态、计算或校验逻辑。
    terms = set(query.lower().split())  # 执行本行的状态、计算或校验逻辑。
    ranked = sorted(messages, key=lambda message: len(terms & set(message.text.lower().split())), reverse=True)  # 执行本行的状态、计算或校验逻辑。
    return ranked[0]  # 执行本行的状态、计算或校验逻辑。
hit = retrieve("订单状态 paid", history)  # 执行本行的状态、计算或校验逻辑。
assert hit.message_id == "m3"  # 执行本行的状态、计算或校验逻辑。
assert hit.role == "tool"  # 执行本行的状态、计算或校验逻辑。
assert "paid" in hit.text  # 执行本行的状态、计算或校验逻辑。


## 8. 指标与制品：压缩率不能取代事实覆盖率

至少同时看 token 节省、pinned 约束保留率、需要时的证据可找回率、摘要版本失配率和最终任务成功率。只看压缩率会鼓励删掉最重要的信息。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
metrics = {"before": context_cost(history), "after": context_cost(selected), "pinned_kept": history[0] in selected, "retrieved": hit.message_id}  # 执行本行的状态、计算或校验逻辑。
artifact_hash = hashlib.sha256(json.dumps(metrics, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert metrics["after"] <= metrics["before"]  # 执行本行的状态、计算或校验逻辑。
assert metrics["pinned_kept"] is True  # 执行本行的状态、计算或校验逻辑。
assert len(artifact_hash) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时依次给出目标、状态合同、动作前校验、主路径、失败分支、指标、制品版本和生产替换点。可靠 Agent 不靠模型自述“完成”，而靠独立的状态 oracle、预算约束、审计和可复放 trace。
